# Modular BERTopic Workflow: Vanilla Captions

This notebook keeps the same analysis idea as the original notebook, but reorganizes the code into reusable functions.

## What the original results suggest

The original run used **124 posts** and produced **6 non-outlier topics** plus a very small outlier group of **2 posts**. That is a good sign for this small dataset because most captions were assigned to interpretable clusters instead of being discarded as noise.

The strongest visible themes were:

| Topic | Size | Main signal from original terms | Early interpretation |
|---:|---:|---|---|
| 0 | 30 | رمضان، فانيلا، فرع، نابلس، بوفيه، الافطار | Ramadan / branch / buffet content |
| 1 | 30 | cake, new, vanilla, coffee, cheesecake, berries | English product-launch and dessert/coffee content |
| 2 | 20 | كيك، فرع، للحجز، الطيرة، الرمضاني | Arabic cake / booking / branch / Ramadan content |
| 3 | 19 | vanilla, coffee, cream, taste, favorite | Mixed English coffee/dessert preference content |
| 4 | 15 | اليوم، فرعنا، لقمة، البوفيه، مشاريب | Brand/location and today/everyday style posts |
| 5 | 8 | كيكه، كريم، matcha-like / premium examples | Smaller niche product group |

From the original summary table:
- Topic **1** had the highest median likes among the larger topics.
- Topic **4** and **5** had the highest median views, but Topic 5 was small, so treat it as exploratory.
- Topics **2**, **3**, and **4** had stronger comment activity, possibly because longer Arabic captions and campaign-style posts invite more interaction.
- The dataset is small, so recommendations should be generated dynamically from metrics instead of hard-coded to one run.

In [ ]:
# Optional installation cell
# Run this only if your environment does not already have these packages.
# %pip install bertopic sentence-transformers umap-learn hdbscan matplotlib arabic-reshaper python-bidi

In [2]:
from dataclasses import dataclass
from pathlib import Path
import ast
import math
import re
import warnings

import numpy as np
import pandas as pd

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import MinMaxScaler
from umap import UMAP
from hdbscan import HDBSCAN

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 300)

In [3]:
@dataclass
class TopicPipelineConfig:
    # Data
    data_path: str = "../data/vanilla_processed.json"
    text_col: str = "caption_text"
    hashtags_col: str = "hashtags"

    # Model cache
    local_model_dir: str = "./.local_models/sentence_transformers/all-MiniLM-L6-v2"
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Vectorizer
    ngram_sizes: tuple = (1, 2, 3)   # exact 1-gram, exact 2-gram, exact 3-gram
    base_ngram_size: int = 1
    min_df: int = 2
    token_pattern: str = r"(?u)\b[^\W\d_][^\W_]+\b"
    top_n_words: int = 20

    # BERTopic / clustering
    n_neighbors: int = 10
    n_components: int = 5
    min_dist: float = 0.0
    umap_metric: str = "cosine"
    min_cluster_size: int = 8
    min_samples: int = 2
    hdbscan_metric: str = "euclidean"
    cluster_selection_method: str = "eom"
    min_topic_size: int = 8
    calculate_probabilities: bool = True
    random_state: int = 42
    language: str = "multilingual"
    verbose: bool = True

    # Recommendation settings
    min_topic_posts_for_strong_claim: int = 8
    sample_docs_per_topic: int = 2
    recommendation_top_k: int = 3

In [4]:
def load_dataset(config: TopicPipelineConfig) -> pd.DataFrame:
    """Load a dataset from JSON or CSV using the configured path."""
    path = Path(config.data_path)

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find data file: {path.resolve()}\\n"
            "Update config.data_path to your dataset location."
        )

    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in [".csv", ".txt"]:
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file type: {path.suffix}. Use JSON or CSV.")

In [5]:
def normalize_arabic_text(text: str) -> str:
    """Normalize frequent Arabic spelling variants and remove diacritics/tatweel."""
    text = str(text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    return text


def _parse_hashtag_tokens(val):
    """Parse hashtag values whether they are lists, stringified lists, comma text, pipe text, or raw hashtag text."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []

    if isinstance(val, list):
        raw = val
    else:
        s = str(val).strip()
        if not s:
            return []

        try:
            parsed = ast.literal_eval(s)
            raw = parsed if isinstance(parsed, list) else re.findall(r"(?<!\w)#([^\s#]+)", s, flags=re.UNICODE)
        except Exception:
            if "|" in s:
                raw = s.split("|")
            elif "," in s:
                raw = s.split(",")
            else:
                raw = re.findall(r"(?<!\w)#([^\s#]+)", s, flags=re.UNICODE)

    out = []
    for x in raw:
        t = str(x).strip().lower().lstrip("#")
        t = normalize_arabic_text(t)
        t = re.sub(r"[^\w\u0600-\u06FF]+", "", t)
        if t:
            out.append(t)

    return list(dict.fromkeys(out))


def clean_caption(text, hashtags_value=None):
    """Clean Arabic/English social caption text and remove explicit hashtag tokens."""
    text = str(text).lower()

    # Remove URLs, mentions, hashtags, long phone-like numbers
    text = re.sub(r"http\S+|www\S+|@\w+", " ", text)
    text = re.sub(r"#\S+", " ", text)
    text = re.sub(r"\b\d{5,}\b", " ", text)

    text = normalize_arabic_text(text)
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    tag_terms = _parse_hashtag_tokens(hashtags_value)
    for term in tag_terms:
        text = re.sub(rf"(?<!\w){re.escape(term)}(?!\w)", " ", text)
        for part in term.split("_"):
            if len(part) >= 3:
                text = re.sub(rf"(?<!\w){re.escape(part)}(?!\w)", " ", text)

    return re.sub(r"\s+", " ", text).strip()


def prepare_texts(df: pd.DataFrame, config: TopicPipelineConfig) -> tuple[pd.DataFrame, list[str]]:
    """Create clean_caption and return texts for modeling."""
    if config.text_col not in df.columns:
        raise KeyError(f"Text column '{config.text_col}' not found. Available columns: {list(df.columns)}")

    df = df.copy()
    raw_text = df[config.text_col].fillna("")

    if config.hashtags_col in df.columns:
        df["clean_caption"] = [
            clean_caption(t, h)
            for t, h in zip(raw_text, df[config.hashtags_col])
        ]
    else:
        df["clean_caption"] = raw_text.apply(lambda x: clean_caption(x, None))

    texts = df["clean_caption"].fillna("").astype(str).tolist()
    return df, texts

In [6]:
def build_stop_words(extra_noise_words=None):
    """Build Arabic + English + project noise stop words."""
    arabic_stop_words = [
        "في", "من", "على", "الى", "إلى", "عن", "مع", "هذا", "هذه", "ذلك",
        "هو", "هي", "هم", "ما", "لا", "نعم", "او", "أو", "كل", "تم",
        "الله", "ان", "إن", "أن", "كان", "كانت", "بعد", "قبل",
    ]

    default_noise_words = [
        "2024", "2025", "2026",
        "gaza",  # remove only if location is not your goal
    ]

    if extra_noise_words is None:
        extra_noise_words = default_noise_words

    return list(set(arabic_stop_words).union(ENGLISH_STOP_WORDS).union(extra_noise_words))


def build_vectorizer(config: TopicPipelineConfig, ngram_size: int = 1, extra_noise_words=None) -> CountVectorizer:
    """Create a CountVectorizer with exact ngram size: 1 only, 2 only, or 3 only."""
    if ngram_size not in [1, 2, 3]:
        raise ValueError("ngram_size must be exactly 1, 2, or 3.")

    return CountVectorizer(
        stop_words=build_stop_words(extra_noise_words=extra_noise_words),
        ngram_range=(ngram_size, ngram_size),
        min_df=config.min_df,
        token_pattern=config.token_pattern,
    )

In [7]:
def load_embedding_model(config: TopicPipelineConfig) -> SentenceTransformer:
    """Load sentence-transformer model from local cache if available, otherwise download and save."""
    local_dir = Path(config.local_model_dir)
    local_dir.parent.mkdir(parents=True, exist_ok=True)

    if local_dir.exists() and any(local_dir.iterdir()):
        model = SentenceTransformer(str(local_dir), local_files_only=True)
    else:
        model = SentenceTransformer(config.embedding_model_name)
        model.save(str(local_dir))

    print(f"Using embedding model from: {local_dir.resolve()}")
    return model


def build_topic_model(
    config: TopicPipelineConfig,
    embedding_model,
    ngram_size: int = 1,
    extra_noise_words=None,
) -> BERTopic:
    """Create a BERTopic model for one exact ngram size."""
    vectorizer_model = build_vectorizer(config, ngram_size=ngram_size, extra_noise_words=extra_noise_words)

    umap_model = UMAP(
        n_neighbors=config.n_neighbors,
        n_components=config.n_components,
        min_dist=config.min_dist,
        metric=config.umap_metric,
        random_state=config.random_state,
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=config.min_cluster_size,
        min_samples=config.min_samples,
        metric=config.hdbscan_metric,
        cluster_selection_method=config.cluster_selection_method,
        prediction_data=True,
    )

    return BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        language=config.language,
        min_topic_size=config.min_topic_size,
        calculate_probabilities=config.calculate_probabilities,
        verbose=config.verbose,
        top_n_words=config.top_n_words,
    )

In [8]:
def fit_topic_model(
    texts: list[str],
    config: TopicPipelineConfig,
    embedding_model,
    ngram_size: int = 1,
    extra_noise_words=None,
):
    """Fit one BERTopic model and return model, topics, probabilities, and topic info."""
    topic_model = build_topic_model(
        config=config,
        embedding_model=embedding_model,
        ngram_size=ngram_size,
        extra_noise_words=extra_noise_words,
    )

    topics, probs = topic_model.fit_transform(texts)
    topic_info = topic_model.get_topic_info()

    print("Exact ngram size:", ngram_size)
    print("Unique topics:", sorted(set(topics)))
    print(topic_info[["Topic", "Count", "Name"]])

    return topic_model, topics, probs, topic_info


def fit_ngram_models(
    texts: list[str],
    config: TopicPipelineConfig,
    embedding_model,
    ngram_sizes=(1, 2, 3),
    extra_noise_words=None,
):
    """
    Fit separate BERTopic models for exact 1-gram, exact 2-gram, and exact 3-gram representations.

    Note: these are separate BERTopic runs, so topic IDs may not perfectly align across ngram sizes.
    For aligned terms by the same base topics, use extract_exact_ngram_terms_by_topic().
    """
    results = {}

    for n in ngram_sizes:
        model, topics, probs, info = fit_topic_model(
            texts=texts,
            config=config,
            embedding_model=embedding_model,
            ngram_size=n,
            extra_noise_words=extra_noise_words,
        )
        results[n] = {
            "model": model,
            "topics": topics,
            "probs": probs,
            "topic_info": info,
        }

    return results

In [9]:
def get_topic_terms(topic_model: BERTopic, topic_id: int, top_n: int = 20) -> list[tuple[str, float]]:
    """Return clean topic terms without empty strings."""
    terms = topic_model.get_topic(topic_id)
    if not terms:
        return []

    return [
        (term, float(score))
        for term, score in terms
        if isinstance(term, str) and term.strip()
    ][:top_n]


def get_all_topic_terms(topic_model: BERTopic, top_n: int = 20) -> pd.DataFrame:
    """Return a long table of terms for each topic from a fitted BERTopic model."""
    rows = []
    info = topic_model.get_topic_info()

    for topic_id in info["Topic"]:
        if topic_id == -1:
            continue
        for rank, (term, score) in enumerate(get_topic_terms(topic_model, topic_id, top_n=top_n), start=1):
            rows.append({
                "topic_id": topic_id,
                "rank": rank,
                "term": term,
                "score": score,
            })

    return pd.DataFrame(rows)

In [10]:
def extract_exact_ngram_terms_by_topic(
    texts: list[str],
    topics: list[int],
    config: TopicPipelineConfig,
    ngram_sizes=(1, 2, 3),
    top_n: int = 15,
    extra_noise_words=None,
) -> pd.DataFrame:
    """
    Extract highest exact 1-, 2-, and 3-gram terms for the same topic assignments.

    This aligns all term extraction to one base topic model. It uses a class-based TF-IDF style score:
    - Combine all docs inside each topic.
    - Vectorize each topic's combined documents.
    - Compute term frequency in topic adjusted by inverse topic frequency.
    """
    df_topic_texts = pd.DataFrame({"text": texts, "topic_id": topics})
    df_topic_texts = df_topic_texts[df_topic_texts["topic_id"] != -1].copy()

    grouped = (
        df_topic_texts
        .groupby("topic_id")["text"]
        .apply(lambda s: " ".join(s.astype(str)))
        .reset_index()
        .sort_values("topic_id")
    )

    rows = []

    for n in ngram_sizes:
        vectorizer = build_vectorizer(config, ngram_size=n, extra_noise_words=extra_noise_words)
        X = vectorizer.fit_transform(grouped["text"])
        terms = np.array(vectorizer.get_feature_names_out())

        tf = X.toarray().astype(float)
        topic_lengths = tf.sum(axis=1, keepdims=True)
        topic_lengths[topic_lengths == 0] = 1
        tf_norm = tf / topic_lengths

        df_term = (tf > 0).sum(axis=0)
        idf = np.log((1 + tf.shape[0]) / (1 + df_term)) + 1
        scores = tf_norm * idf

        for row_idx, topic_id in enumerate(grouped["topic_id"].tolist()):
            if scores.shape[1] == 0:
                continue

            top_indices = np.argsort(scores[row_idx])[::-1][:top_n]
            for rank, term_idx in enumerate(top_indices, start=1):
                if scores[row_idx, term_idx] <= 0:
                    continue
                rows.append({
                    "topic_id": topic_id,
                    "ngram_size": n,
                    "rank": rank,
                    "term": terms[term_idx],
                    "score": float(scores[row_idx, term_idx]),
                })

    return pd.DataFrame(rows)


def pivot_top_ngram_terms(ngram_terms: pd.DataFrame, top_n_per_ngram: int = 8) -> pd.DataFrame:
    """Create a readable topic table with top 1-, 2-, and 3-gram terms side by side."""
    if ngram_terms.empty:
        return pd.DataFrame()

    filtered = ngram_terms[ngram_terms["rank"] <= top_n_per_ngram].copy()

    out = (
        filtered
        .sort_values(["topic_id", "ngram_size", "rank"])
        .groupby(["topic_id", "ngram_size"])["term"]
        .apply(lambda s: ", ".join(s.astype(str)))
        .unstack("ngram_size")
        .reset_index()
        .rename(columns={1: "top_1grams", 2: "top_2grams", 3: "top_3grams"})
    )

    return out

In [11]:
def attach_topics(df: pd.DataFrame, topics: list[int], topic_model: BERTopic) -> pd.DataFrame:
    """Attach topic IDs and topic names to the dataset."""
    out = df.copy()
    out["topic_id"] = topics

    topic_info = topic_model.get_topic_info()
    out = out.merge(
        topic_info[["Topic", "Name"]],
        left_on="topic_id",
        right_on="Topic",
        how="left",
    ).drop(columns=["Topic"])

    return out


def get_topic_samples(
    df_with_topics: pd.DataFrame,
    config: TopicPipelineConfig,
    sample_cols=None,
) -> pd.DataFrame:
    """Sample representative rows from each non-outlier topic."""
    if sample_cols is None:
        sample_cols = [
            "topic_id",
            config.text_col,
            "clean_caption",
            "view_rate",
            "engagement_rate",
            "likes_count",
            "comments_count",
            "views_count",
        ]

    rows = []

    for topic_id, group in df_with_topics[df_with_topics["topic_id"] != -1].groupby("topic_id"):
        sample = group.sample(min(config.sample_docs_per_topic, len(group)), random_state=config.random_state)
        rows.append(sample)

    if not rows:
        return pd.DataFrame(columns=[c for c in sample_cols if c in df_with_topics.columns])

    topic_samples = pd.concat(rows, ignore_index=True)
    sample_cols = [c for c in sample_cols if c in topic_samples.columns]

    return topic_samples.sort_values("topic_id")[sample_cols]

In [12]:
def detect_numeric_metric_columns(df: pd.DataFrame) -> list[str]:
    """
    Dynamically detect useful numeric metric columns.

    This keeps the recommendations reusable for datasets that may have:
    likes, comments, views, saves, shares, engagement_rate, view_rate, caption_length, etc.
    """
    candidate_keywords = [
        "like", "comment", "view", "engagement", "rate", "share", "save",
        "reach", "impression", "click", "caption_length", "hashtag", "emoji",
    ]

    numeric_cols = []

    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        col_lower = col.lower()
        if any(k in col_lower for k in candidate_keywords):
            numeric_cols.append(col)

    return numeric_cols


def summarize_topic_performance(df_with_topics: pd.DataFrame) -> pd.DataFrame:
    """Build a dynamic topic performance summary from available numeric metrics."""
    df_topics = df_with_topics[df_with_topics["topic_id"] != -1].copy()
    numeric_cols = detect_numeric_metric_columns(df_topics)

    if df_topics.empty:
        return pd.DataFrame()

    agg_spec = {"posts": ("topic_id", "size")}

    for col in numeric_cols:
        agg_spec[f"median_{col}"] = (col, "median")
        agg_spec[f"mean_{col}"] = (col, "mean")

    summary = (
        df_topics
        .groupby("topic_id")
        .agg(**agg_spec)
        .reset_index()
        .sort_values("posts", ascending=False)
    )

    return summary


def choose_primary_performance_metrics(topic_summary: pd.DataFrame) -> list[str]:
    """Choose metrics to score topics, preferring engagement/rates when available."""
    if topic_summary.empty:
        return []

    cols = topic_summary.columns.tolist()
    preferred_order = [
        "median_engagement_rate",
        "mean_engagement_rate",
        "median_view_rate",
        "mean_view_rate",
        "median_likes_count",
        "median_comments_count",
        "median_views_count",
        "mean_likes_count",
        "mean_comments_count",
        "mean_views_count",
    ]

    selected = [c for c in preferred_order if c in cols]

    if selected:
        return selected[:4]

    fallback = [
        c for c in cols
        if c != "posts" and (c.startswith("median_") or c.startswith("mean_"))
    ]
    return fallback[:4]


def add_dynamic_topic_scores(topic_summary: pd.DataFrame) -> pd.DataFrame:
    """
    Add a performance_score using whatever metrics exist.

    Higher score = stronger observed performance relative to the other topics.
    """
    if topic_summary.empty:
        return topic_summary

    out = topic_summary.copy()
    metric_cols = choose_primary_performance_metrics(out)

    if not metric_cols:
        out["performance_score"] = np.nan
        out["support_score"] = out["posts"] / max(out["posts"].max(), 1)
        out["opportunity_score"] = out["support_score"]
        return out

    clean_metrics = out[metric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(clean_metrics)

    out["performance_score"] = scaled.mean(axis=1)
    out["support_score"] = out["posts"] / max(out["posts"].max(), 1)
    out["opportunity_score"] = 0.7 * out["performance_score"] + 0.3 * out["support_score"]
    out["scored_metrics"] = ", ".join(metric_cols)

    return out.sort_values("opportunity_score", ascending=False)

In [13]:
def make_dynamic_recommendations(
    scored_summary: pd.DataFrame,
    ngram_pivot: pd.DataFrame | None = None,
    config: TopicPipelineConfig | None = None,
) -> pd.DataFrame:
    """
    Generate recommendations from topic size + available performance metrics.

    The logic is dataset-agnostic:
    - Scale topics with high performance and enough posts.
    - Test topics with high performance but low support.
    - Improve/reframe topics with high support but weak performance.
    - Monitor tiny topics instead of overclaiming.
    """
    if config is None:
        config = TopicPipelineConfig()

    if scored_summary.empty:
        return pd.DataFrame(columns=["topic_id", "action", "reason", "suggested_next_step"])

    df = scored_summary.copy()

    if ngram_pivot is not None and not ngram_pivot.empty:
        df = df.merge(ngram_pivot, on="topic_id", how="left")

    perf_q75 = df["performance_score"].quantile(0.75) if "performance_score" in df else np.nan
    perf_q25 = df["performance_score"].quantile(0.25) if "performance_score" in df else np.nan
    support_median = df["posts"].median()

    recs = []

    for _, row in df.iterrows():
        topic_id = row["topic_id"]
        posts = row["posts"]
        performance = row.get("performance_score", np.nan)
        terms = row.get("top_2grams", None) or row.get("top_1grams", None) or ""
        terms = str(terms)[:180]

        if posts >= support_median and performance >= perf_q75:
            action = "Scale"
            reason = "This topic has both solid support and above-average performance."
            next_step = f"Create more content around the strongest repeated language/themes: {terms}"
        elif posts < support_median and performance >= perf_q75:
            action = "Test more"
            reason = "This topic performs well but has fewer posts, so confidence is lower."
            next_step = f"Run a small content test with 3-5 new posts using related 2-gram/3-gram phrases: {terms}"
        elif posts >= support_median and performance <= perf_q25:
            action = "Rework"
            reason = "This topic appears often but underperforms relative to other topics."
            next_step = "Change the hook, creative format, CTA, or posting time before increasing volume."
        elif posts < config.min_topic_posts_for_strong_claim:
            action = "Monitor"
            reason = "The topic is small, so avoid strong conclusions."
            next_step = "Collect more examples before making a major decision."
        else:
            action = "Maintain"
            reason = "This topic is around the middle of the dataset."
            next_step = "Keep it in the mix, but prioritize topics with stronger opportunity scores."

        recs.append({
            "topic_id": topic_id,
            "posts": posts,
            "performance_score": performance,
            "opportunity_score": row.get("opportunity_score", np.nan),
            "action": action,
            "reason": reason,
            "suggested_next_step": next_step,
        })

    return pd.DataFrame(recs).sort_values(["opportunity_score"], ascending=False)

In [14]:
def visualize_safe_topics(topic_model: BERTopic):
    """Avoid visualization errors when there are too few non-outlier topics."""
    info = topic_model.get_topic_info()
    non_outlier_topics = info[info["Topic"] != -1]["Topic"].tolist()

    print("Non-outlier topics:", non_outlier_topics)

    if len(non_outlier_topics) >= 3:
        return topic_model.visualize_topics(topics=non_outlier_topics)

    print("Not enough non-outlier topics for intertopic map; showing bar chart instead.")
    return topic_model.visualize_barchart(
        topics=non_outlier_topics if non_outlier_topics else None,
        top_n_topics=len(non_outlier_topics) or 1,
    )


def export_outputs(
    df_with_topics: pd.DataFrame,
    topic_summary: pd.DataFrame,
    ngram_terms: pd.DataFrame,
    recommendations: pd.DataFrame,
    output_dir: str = "./topic_outputs",
):
    """Export reusable outputs to CSV."""
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    df_with_topics.to_csv(out / "dataset_with_topics.csv", index=False)
    topic_summary.to_csv(out / "topic_summary.csv", index=False)
    ngram_terms.to_csv(out / "topic_terms_exact_1_2_3grams.csv", index=False)
    recommendations.to_csv(out / "dynamic_recommendations.csv", index=False)

    print(f"Saved outputs to: {out.resolve()}")

## Run the pipeline

The next cells execute the modular workflow.

Change only the `TopicPipelineConfig(...)` values when you want to reuse this notebook on another dataset.

In [15]:
config = TopicPipelineConfig(
    data_path="../data/vanilla_processed.json",
    text_col="caption_text",
    hashtags_col="hashtags",

    # Exact ngram options requested:
    ngram_sizes=(1, 2, 3),
    base_ngram_size=1,

    # Keep the original modeling behavior:
    min_df=2,
    min_cluster_size=8,
    min_samples=2,
    min_topic_size=8,
    top_n_words=20,
)

df = load_dataset(config)
df, texts = prepare_texts(df, config)

print("Rows:", len(df))
print("Texts:", len(texts))
df.head()

Rows: 124
Texts: 124


,business_name,sector,followers_count,post_date,day_of_week,month,post_type,caption_text,caption_length,hashtags_count,...,views_count,language,CTA_present,promo_post,mentions_location,religious_theme,patriotic_theme,arabic_dialect_style,sponsored,clean_caption
0,Vanilla Palestine,Cafes/Restaurants,139000,2025-02-05,Wednesday,February,reel,شاركونا إجابتكم بالتعليقات 👇 #فانيلا_اليوم_وكل_يوم #vanilla_today_and_everyday,83,2,...,15200,Arabic,True,False,False,False,False,True,0,شاركونا اجابتكم بالتعليقات
1,Vanilla Palestine,Cafes/Restaurants,139000,2025-02-04,Tuesday,February,reel,What’s your favorite cake ❤️🍰? #VANILLA_Today_and_everyday,65,1,...,11000,English,True,False,False,False,False,False,0,what s your favorite cake
2,Vanilla Palestine,Cafes/Restaurants,139000,2025-02-01,Saturday,February,reel,كل يوم له بداية… وأجمل بداية بطاقة إيجابية #فانيلا_اليوم_وكل_يوم,72,1,...,12500,Arabic,False,False,False,False,False,True,0,كل يوم له بدايه واجمل بدايه بطاقه ايجابيه
3,Vanilla Palestine,Cafes/Restaurants,139000,2025-01-24,Friday,January,reel,ومن أهم فوائدها إنها تذهب الحزن.. التلبينة النبوية.. صحية، فوائدها كثيرة ولذيذة هي طبق تقليدي قديم ذكرها الرسول محمد صلى الله عليه وسلم في أحاديث لها فيها من فوائد وشفاء #فانيلا_اليوم_وكل_يوم #vanilla_today_and_everyday,239,2,...,214000,Arabic,False,False,False,True,False,True,0,ومن اهم فوايدها انها تذهب الحزن التلبينه النبويه صحيه، فوايدها كثيره ولذيذه هي طبق تقليدي قديم ذكرها الرسول محمد صلي الله عليه وسلم في احاديث لها فيها من فوايد وشفاء
4,Vanilla Palestine,Cafes/Restaurants,139000,2023-10-03,Tuesday,October,reel,استمر بالأكل أنت بأمان 😜🤤 #فانيلا_اليوم_وكل_يوم #طعام #وزن #أكل #كيك VANILLA #Today_and_Everyday #Food #Eating #Cake #Desserts,142,10,...,78200,Mixed,False,False,False,False,False,True,0,استمر بالاكل انت بامان vanilla


In [16]:
embedding_model = load_embedding_model(config)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10301.24it/s]

Using embedding model from: C:\univ\Data mining\Project\notebooks\.local_models\sentence_transformers\all-MiniLM-L6-v2


In [17]:
# Base model: exact unigram vectorizer, matching the original notebook behavior.
base_topic_model, base_topics, base_probs, base_topic_info = fit_topic_model(
    texts=texts,
    config=config,
    embedding_model=embedding_model,
    ngram_size=config.base_ngram_size,
)

base_topic_info

2026-05-15 01:04:53,443 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 4/4 [00:01<00:00,  3.09it/s]
2026-05-15 01:04:54,751 - BERTopic - Embedding - Completed ✓
2026-05-15 01:04:54,752 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 01:05:04,238 - BERTopic - Dimensionality - Completed ✓
2026-05-15 01:05:04,239 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 01:05:04,246 - BERTopic - Cluster - Completed ✓
2026-05-15 01:05:04,249 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 01:05:04,260 - BERTopic - Representation - Completed ✓


Exact ngram size: 1
Unique topics: [-1, 0, 1, 2, 3, 4, 5]
   Topic  Count                       Name
0     -1      2         -1_علي_ام_مين_انها
1      0     30      0_رمضان_فانيلا_شو_فرع
2      1     30  1_cake_new_vanilla_coffee
3      2     20      2_الان_فانيلا_كيك_فرع
4      3     19   3_vanilla_فقط_coffee_وكل
5      4     15     4_وكل_اليوم_يوم_فانيلا
6      5      8      5_اكبر_like_كريم_كيكه


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2,-1_علي_ام_مين_انها,"[علي, ام, مين, انها, فانيلا, , , , , , , , , , , , , , , ]","[مين جاهز للحلويات الرمضانيه؟, كلكم اتفقتوا علي انها اطيب ام علي في فانيلا]"
1,0,30,0_رمضان_فانيلا_شو_فرع,"[رمضان, فانيلا, شو, فرع, نابلس, ام, ramadan, ايام, شاركونا, البوفيه, بالتعليقات, فيكم, مشاريب, كريم, فرنش, عجيب, الافطار, توست, بوفيه, الرمضاني]","[هذا ما نقدمه لكم حلي رمضان النهايي, 16 رمضان, رمضان فانيلا]"
2,1,30,1_cake_new_vanilla_coffee,"[cake, new, vanilla, coffee, freshly, cheesecake, favorite, berries, cream, hot, taste, , , , , , , , , ]","[dream vanilla cake creamy and yummy, new mix berries cake the cake you always dreamed about is now available at all vanilla branches, the new very chocolate cake]"
3,2,20,2_الان_فانيلا_كيك_فرع,"[الان, فانيلا, كيك, فرع, كيكه, للحجز, وفرع, الطيره, الرمضاني, بوفيه, يوم, تشيزكيك, عجيب, سان, نابلس, الافطار, توست, شو, علي, اليوم]","[بوفيه الافطار الرمضاني في فانيلا فرع الطيره وفرع نابلس للحجز او عبر رسايل الصفحه نستقبل المكالمات يوميا بعد الساعه الواحده ظهرا, بوفيه الافطار الرمضاني في فانيلا فرع الطيره وفرع نابلس للحجز او عبر رسايل الصفحه نستقبل المكالمات يوميا بعد الساعه الواحده ظهرا, نستقبل حجوزاتكم الان بوفيه فانيلا الر..."
4,3,19,3_vanilla_فقط_coffee_وكل,"[vanilla, فقط, coffee, وكل, اليوم, علي, cream, الكراميل, taste, like, شاركونا, favorite, ramadan, hot, berries, فيكم, مين, بمناسبه, انت, يوم]","[سعيد صباحكم من فانيلا vanilla, طول بالك لما تتعب بالزينه بالاخر يغير رايه الزبون ويصير بده اياها برا vanilla, شرب القهوه بالليل يوذي المعده؟ vanilla]"
5,4,15,4_وكل_اليوم_يوم_فانيلا,"[وكل, اليوم, يوم, فانيلا, الطيره, نابلس, انت, بمناسبه, فرعنا, لقمه, الصحيح, البوفيه, مشاريب, انها, وفرع, الافطار, للحجز, الرمضاني, بوفيه, فرع]","[نقطه ضعف بطعم الفيريرو فانيلا اليوم وكل يوم, بتشاركوهم ولا بتخلصهم لحالك؟ share it or all yours ؟ فانيلا اليوم وكل يوم, فانيلا الها حكايه والحكايه بلشت من هون فانيلا اليوم وكل يوم]"
6,5,8,5_اكبر_like_كريم_كيكه,"[اكبر, like, كريم, كيكه, كيك, نابلس, يوم, فانيلا, , , , , , , , , , , , ]","[كارميل كرانش كريم كيكه جديده من فانيلا, اكبر كميه كوكيز, اكبر كميه لفرع نابلس]"


In [18]:
# Optional: run the full BERTopic process separately for exact 1-gram, 2-gram, and 3-gram vectorizers.
# This is useful if you want to compare how topic representation changes with different phrase lengths.
# Topic IDs across these separate models are not guaranteed to align.

ngram_model_results = fit_ngram_models(
    texts=texts,
    config=config,
    embedding_model=embedding_model,
    ngram_sizes=config.ngram_sizes,
)

2026-05-15 01:05:11,754 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 4/4 [00:01<00:00,  2.98it/s]
2026-05-15 01:05:13,104 - BERTopic - Embedding - Completed ✓
2026-05-15 01:05:13,104 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 01:05:13,321 - BERTopic - Dimensionality - Completed ✓
2026-05-15 01:05:13,321 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 01:05:13,329 - BERTopic - Cluster - Completed ✓
2026-05-15 01:05:13,332 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 01:05:13,341 - BERTopic - Representation - Completed ✓
2026-05-15 01:05:13,360 - BERTopic - Embedding - Transforming documents to embeddings.


Exact ngram size: 1
Unique topics: [-1, 0, 1, 2, 3, 4, 5]
   Topic  Count                       Name
0     -1      2         -1_علي_ام_مين_انها
1      0     30      0_رمضان_فانيلا_شو_فرع
2      1     30  1_cake_new_vanilla_coffee
3      2     20      2_الان_فانيلا_كيك_فرع
4      3     19   3_vanilla_فقط_coffee_وكل
5      4     15     4_وكل_اليوم_يوم_فانيلا
6      5      8      5_اكبر_like_كريم_كيكه


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.21it/s]
2026-05-15 01:05:14,610 - BERTopic - Embedding - Completed ✓
2026-05-15 01:05:14,611 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 01:05:14,831 - BERTopic - Dimensionality - Completed ✓
2026-05-15 01:05:14,831 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 01:05:14,839 - BERTopic - Cluster - Completed ✓
2026-05-15 01:05:14,841 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 01:05:14,849 - BERTopic - Representation - Completed ✓
2026-05-15 01:05:14,869 - BERTopic - Embedding - Transforming documents to embeddings.


Exact ngram size: 2
Unique topics: [-1, 0, 1, 2, 3, 4, 5]
   Topic  Count                                                  Name
0     -1      2                                                -1____
1      0     30  0_فانيلا فانيلا_علي البوفيه_توست فانيلا_بوفيه فانيلا
2      1     30           1_favorite cake_vanilla taste_vanilla cake_
3      2     20       2_فرع الطيره_نابلس للحجز_وفرع نابلس_الطيره وفرع
4      3     19  3_vanilla taste_vanilla cake_favorite cake_اليوم وكل
5      4     15          4_وكل يوم_فانيلا اليوم_اليوم وكل_علي البوفيه
6      5      8                                                 5____


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.20it/s]
2026-05-15 01:05:16,124 - BERTopic - Embedding - Completed ✓
2026-05-15 01:05:16,124 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 01:05:16,336 - BERTopic - Dimensionality - Completed ✓
2026-05-15 01:05:16,337 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 01:05:16,344 - BERTopic - Cluster - Completed ✓
2026-05-15 01:05:16,347 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 01:05:16,353 - BERTopic - Representation - Completed ✓


Exact ngram size: 3
Unique topics: [-1, 0, 1, 2, 3, 4, 5]
   Topic  Count  \
0     -1      2   
1      0     30   
2      1     30   
3      2     20   
4      3     19   
5      4     15   
6      5      8   

                                                                              Name  
0                                                                           -1____  
1                                                            0_فرنش توست فانيلا___  
2                                                                            1____  
3     2_الطيره وفرع نابلس_فرع الطيره وفرع_وفرع نابلس للحجز_الافطار الرمضاني فانيلا  
4                                               3_اليوم وكل يوم_فانيلا اليوم وكل__  
5  4_فانيلا اليوم وكل_اليوم وكل يوم_بوفيه الافطار الرمضاني_الافطار الرمضاني فانيلا  
6                                                                            5____  


In [19]:
# Aligned exact 1-, 2-, and 3-gram terms for the SAME base topics.
# This is the clearest way to say: "For each topic, what are the strongest unigram, bigram, and trigram labels?"

exact_ngram_terms = extract_exact_ngram_terms_by_topic(
    texts=texts,
    topics=base_topics,
    config=config,
    ngram_sizes=config.ngram_sizes,
    top_n=15,
)

exact_ngram_terms.head(30)

,topic_id,ngram_size,rank,term,score
0,0,1,1,فانيلا,0.325530
1,0,1,2,رمضان,0.236833
2,0,1,3,شو,0.094733
3,0,1,4,فرع,0.079980
4,0,1,5,نابلس,0.059187
5,0,1,6,عجيب,0.047367
6,0,1,7,فيكم,0.047367
7,0,1,8,كريم,0.047367
8,0,1,9,شاركونا,0.047367
9,0,1,10,بالتعليقات,0.047367


In [20]:
topic_ngram_labels = pivot_top_ngram_terms(exact_ngram_terms, top_n_per_ngram=8)
topic_ngram_labels

ngram_size,topic_id,top_1grams,top_2grams,top_3grams
0,0,"فانيلا, رمضان, شو, فرع, نابلس, عجيب, فيكم, كريم","توست فانيلا, بوفيه فانيلا, فانيلا فانيلا, علي البوفيه, فرنش توست, فرع الطيره",فرنش توست فانيلا
1,1,"cake, new, vanilla, coffee, cheesecake, freshly, taste, cream","vanilla cake, vanilla taste, favorite cake",NaN
2,2,"فانيلا, الان, كيك, فرع, كيكه, للحجز, وفرع, يوم","فرع الطيره, نابلس للحجز, وفرع نابلس, الطيره وفرع, الافطار الرمضاني, بوفيه الافطار, فانيلا فرع, الرمضاني فانيلا","وفرع نابلس للحجز, فرع الطيره وفرع, الطيره وفرع نابلس, الافطار الرمضاني فانيلا, بوفيه الافطار الرمضاني, فانيلا فرع الطيره, الرمضاني فانيلا فرع, فرنش توست فانيلا"
3,3,"vanilla, فانيلا, cake, فقط, coffee, اليوم, وكل, يوم","وكل يوم, اليوم وكل, فانيلا اليوم, vanilla taste, vanilla cake, favorite cake, فرنش توست","اليوم وكل يوم, فانيلا اليوم وكل"
4,4,"فانيلا, وكل, اليوم, يوم, الطيره, نابلس, وفرع, مشاريب","وكل يوم, اليوم وكل, فانيلا اليوم, نابلس للحجز, وفرع نابلس, علي البوفيه, فانيلا فرع, الرمضاني فانيلا","اليوم وكل يوم, فانيلا اليوم وكل, بوفيه الافطار الرمضاني, وفرع نابلس للحجز, فانيلا فرع الطيره, فرع الطيره وفرع, الرمضاني فانيلا فرع, الطيره وفرع نابلس"
5,5,"اكبر, كيك, كيكه, كريم, like, يوم, نابلس, فانيلا",NaN,NaN


In [21]:
df_with_topics = attach_topics(df, base_topics, base_topic_model)
topic_samples = get_topic_samples(df_with_topics, config)

topic_samples

,topic_id,caption_text,clean_caption,likes_count,comments_count,views_count
0,0,مكانكم المفضل ❤️ #فانيلا_اليوم_وكل_يوم,مكانكم المفضل,1300,2,44000
1,0,بعد الإفطار في فانيلا 🌙🤍✌️,بعد الافطار في فانيلا,2898,4,82600
2,1,Summer’s calling 📞 Grab your Yafa Refresher at VANILLA! 🍹☀️ #VANILLA_Today_and_everyday,summer s calling grab your yafa refresher at vanilla,1300,0,56000
3,1,New year new me 😍😍😍,new year new me,2155,10,348000
4,2,ومن أهم فوائدها إنها تذهب الحزن.. التلبينة النبوية.. صحية، فوائدها كثيرة ولذيذة هي طبق تقليدي قديم ذكرها الرسول محمد صلى الله عليه وسلم في أحاديث لها فيها من فوائد وشفاء #فانيلا_اليوم_وكل_يوم #vanilla_today_and_everyday,ومن اهم فوايدها انها تذهب الحزن التلبينه النبويه صحيه، فوايدها كثيره ولذيذه هي طبق تقليدي قديم ذكرها الرسول محمد صلي الله عليه وسلم في احاديث لها فيها من فوايد وشفاء,1313,0,214000
5,2,أمي.. كل عام وأنتِ أجمل نعمة 💜 هذا يومك، نقدّر وجودك بكل لحظة من فانيلا 🎂 الحجوزات عبر رسائل الصفحة 💌,امي كل عام وانت اجمل نعمه هذا يومك، نقدر وجودك بكل لحظه من فانيلا الحجوزات عبر رسايل الصفحه,44,4,56000
6,3,استمر بالأكل أنت بأمان 😜🤤 #فانيلا_اليوم_وكل_يوم #طعام #وزن #أكل #كيك VANILLA #Today_and_Everyday #Food #Eating #Cake #Desserts,استمر بالاكل انت بامان vanilla,963,5,78200
7,3,قاعدين على قلوبكم ❤️🥳 #فانيلا_اليوم_وكل_يوم #كيك #قهوة #وينجز #مشاريب VANILLA #Today_and_Everyday #wings #coffee #cake #drinks,قاعدين علي قلوبكم vanilla,1080,7,143000
8,4,طبقات من القشطة و التمر الفاخر 🌙😍,طبقات من القشطه و التمر الفاخر,1475,2,566000
9,4,Perfect Alfajores🤭,perfect alfajores,541,11,462000


In [22]:
topic_summary = summarize_topic_performance(df_with_topics)
scored_topic_summary = add_dynamic_topic_scores(topic_summary)

# Add phrase labels to the summary table
scored_topic_summary_with_terms = scored_topic_summary.merge(topic_ngram_labels, on="topic_id", how="left")

scored_topic_summary_with_terms

,topic_id,posts,median_caption_length,mean_caption_length,median_hashtags_count,mean_hashtags_count,median_emoji_count,mean_emoji_count,median_likes_count,mean_likes_count,...,mean_comments_count,median_views_count,mean_views_count,performance_score,support_score,opportunity_score,scored_metrics,top_1grams,top_2grams,top_3grams
0,1,30,68.0,72.733333,1.0,1.866667,1.5,1.866667,1019.0,1598.866667,...,269.333333,149500.0,251506.666667,0.633646,1.000000,0.743552,"median_likes_count, median_comments_count, median_views_count, mean_likes_count","cake, new, vanilla, coffee, cheesecake, freshly, taste, cream","vanilla cake, vanilla taste, favorite cake",NaN
1,0,30,27.5,33.300000,0.5,0.900000,1.0,1.400000,914.5,1026.933333,...,22.266667,107800.0,181673.333333,0.343779,1.000000,0.540645,"median_likes_count, median_comments_count, median_views_count, mean_likes_count","فانيلا, رمضان, شو, فرع, نابلس, عجيب, فيكم, كريم","توست فانيلا, بوفيه فانيلا, فانيلا فانيلا, علي البوفيه, فرنش توست, فرع الطيره",فرنش توست فانيلا
2,2,20,180.0,198.500000,3.5,2.900000,4.0,4.350000,520.5,634.200000,...,253.700000,122000.0,192950.000000,0.331049,0.666667,0.431734,"median_likes_count, median_comments_count, median_views_count, mean_likes_count","فانيلا, الان, كيك, فرع, كيكه, للحجز, وفرع, يوم","فرع الطيره, نابلس للحجز, وفرع نابلس, الطيره وفرع, الافطار الرمضاني, بوفيه الافطار, فانيلا فرع, الرمضاني فانيلا","وفرع نابلس للحجز, فرع الطيره وفرع, الطيره وفرع نابلس, الافطار الرمضاني فانيلا, بوفيه الافطار الرمضاني, فانيلا فرع الطيره, الرمضاني فانيلا فرع, فرنش توست فانيلا"
3,4,15,84.0,93.866667,2.0,3.066667,1.0,2.000000,557.0,834.733333,...,328.400000,238000.0,337266.666667,0.377003,0.500000,0.413902,"median_likes_count, median_comments_count, median_views_count, mean_likes_count","فانيلا, وكل, اليوم, يوم, الطيره, نابلس, وفرع, مشاريب","وكل يوم, اليوم وكل, فانيلا اليوم, نابلس للحجز, وفرع نابلس, علي البوفيه, فانيلا فرع, الرمضاني فانيلا","اليوم وكل يوم, فانيلا اليوم وكل, بوفيه الافطار الرمضاني, وفرع نابلس للحجز, فانيلا فرع الطيره, فرع الطيره وفرع, الرمضاني فانيلا فرع, الطيره وفرع نابلس"
4,3,19,142.0,153.315789,4.0,5.105263,2.0,2.105263,788.0,781.736842,...,203.947368,136000.0,149442.105263,0.307017,0.633333,0.404912,"median_likes_count, median_comments_count, median_views_count, mean_likes_count","vanilla, فانيلا, cake, فقط, coffee, اليوم, وكل, يوم","وكل يوم, اليوم وكل, فانيلا اليوم, vanilla taste, vanilla cake, favorite cake, فرنش توست","اليوم وكل يوم, فانيلا اليوم وكل"
5,5,8,64.5,62.375000,2.0,2.250000,2.0,1.500000,365.5,979.250000,...,7.125000,271000.0,337312.500000,0.339422,0.266667,0.317595,"median_likes_count, median_comments_count, median_views_count, mean_likes_count","اكبر, كيك, كيكه, كريم, like, يوم, نابلس, فانيلا",NaN,NaN


In [23]:
recommendations = make_dynamic_recommendations(
    scored_summary=scored_topic_summary,
    ngram_pivot=topic_ngram_labels,
    config=config,
)

recommendations

,topic_id,posts,performance_score,opportunity_score,action,reason,suggested_next_step
0,1,30,0.633646,0.743552,Scale,This topic has both solid support and above-average performance.,"Create more content around the strongest repeated language/themes: vanilla cake, vanilla taste, favorite cake"
1,0,30,0.343779,0.540645,Maintain,This topic is around the middle of the dataset.,"Keep it in the mix, but prioritize topics with stronger opportunity scores."
2,2,20,0.331049,0.431734,Rework,This topic appears often but underperforms relative to other topics.,"Change the hook, creative format, CTA, or posting time before increasing volume."
3,4,15,0.377003,0.413902,Test more,"This topic performs well but has fewer posts, so confidence is lower.","Run a small content test with 3-5 new posts using related 2-gram/3-gram phrases: وكل يوم, اليوم وكل, فانيلا اليوم, نابلس للحجز, وفرع نابلس, علي البوفيه, فانيلا فرع, الرمضاني فانيلا"
4,3,19,0.307017,0.404912,Maintain,This topic is around the middle of the dataset.,"Keep it in the mix, but prioritize topics with stronger opportunity scores."
5,5,8,0.339422,0.317595,Maintain,This topic is around the middle of the dataset.,"Keep it in the mix, but prioritize topics with stronger opportunity scores."


## Visualizations

In [30]:
ngram_models = {}
ngram_topics = {}
ngram_probs = {}
ngram_topic_infos = {}

for ngram_size in config.ngram_sizes:
    print(f"Fitting exact {ngram_size}-gram topic model...")

    topic_model, topics, probs, topic_info = fit_topic_model(
        texts=texts,
        config=config,
        embedding_model=embedding_model,
        ngram_size=ngram_size,
    )

    ngram_models[ngram_size] = topic_model
    ngram_topics[ngram_size] = topics
    ngram_probs[ngram_size] = probs
    ngram_topic_infos[ngram_size] = topic_info

print("Finished fitting:", list(ngram_models.keys()))

2026-05-15 01:18:24,152 - BERTopic - Embedding - Transforming documents to embeddings.


Fitting exact 1-gram topic model...


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.33it/s]
2026-05-15 01:18:25,363 - BERTopic - Embedding - Completed ✓
2026-05-15 01:18:25,364 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 01:18:25,592 - BERTopic - Dimensionality - Completed ✓
2026-05-15 01:18:25,592 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 01:18:25,600 - BERTopic - Cluster - Completed ✓
2026-05-15 01:18:25,602 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 01:18:25,613 - BERTopic - Representation - Completed ✓
2026-05-15 01:18:25,638 - BERTopic - Embedding - Transforming documents to embeddings.


Exact ngram size: 1
Unique topics: [-1, 0, 1, 2, 3, 4, 5]
   Topic  Count                       Name
0     -1      2         -1_علي_ام_مين_انها
1      0     30      0_رمضان_فانيلا_شو_فرع
2      1     30  1_cake_new_vanilla_coffee
3      2     20      2_الان_فانيلا_كيك_فرع
4      3     19   3_vanilla_فقط_coffee_وكل
5      4     15     4_وكل_اليوم_يوم_فانيلا
6      5      8      5_اكبر_like_كريم_كيكه
Fitting exact 2-gram topic model...


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.19it/s]
2026-05-15 01:18:26,901 - BERTopic - Embedding - Completed ✓
2026-05-15 01:18:26,901 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 01:18:27,121 - BERTopic - Dimensionality - Completed ✓
2026-05-15 01:18:27,121 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 01:18:27,128 - BERTopic - Cluster - Completed ✓
2026-05-15 01:18:27,131 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 01:18:27,138 - BERTopic - Representation - Completed ✓
2026-05-15 01:18:27,158 - BERTopic - Embedding - Transforming documents to embeddings.


Exact ngram size: 2
Unique topics: [-1, 0, 1, 2, 3, 4, 5]
   Topic  Count                                                  Name
0     -1      2                                                -1____
1      0     30  0_فانيلا فانيلا_علي البوفيه_توست فانيلا_بوفيه فانيلا
2      1     30           1_favorite cake_vanilla taste_vanilla cake_
3      2     20       2_فرع الطيره_نابلس للحجز_وفرع نابلس_الطيره وفرع
4      3     19  3_vanilla taste_vanilla cake_favorite cake_اليوم وكل
5      4     15          4_وكل يوم_فانيلا اليوم_اليوم وكل_علي البوفيه
6      5      8                                                 5____
Fitting exact 3-gram topic model...


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.36it/s]
2026-05-15 01:18:28,355 - BERTopic - Embedding - Completed ✓
2026-05-15 01:18:28,356 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 01:18:28,583 - BERTopic - Dimensionality - Completed ✓
2026-05-15 01:18:28,583 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 01:18:28,591 - BERTopic - Cluster - Completed ✓
2026-05-15 01:18:28,594 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 01:18:28,602 - BERTopic - Representation - Completed ✓


Exact ngram size: 3
Unique topics: [-1, 0, 1, 2, 3, 4, 5]
   Topic  Count  \
0     -1      2   
1      0     30   
2      1     30   
3      2     20   
4      3     19   
5      4     15   
6      5      8   

                                                                              Name  
0                                                                           -1____  
1                                                            0_فرنش توست فانيلا___  
2                                                                            1____  
3     2_الطيره وفرع نابلس_فرع الطيره وفرع_وفرع نابلس للحجز_الافطار الرمضاني فانيلا  
4                                               3_اليوم وكل يوم_فانيلا اليوم وكل__  
5  4_فانيلا اليوم وكل_اليوم وكل يوم_بوفيه الافطار الرمضاني_الافطار الرمضاني فانيلا  
6                                                                            5____  
Finished fitting: [1, 2, 3]


In [32]:
visualize_safe_topics(ngram_models[1])

Non-outlier topics: [0, 1, 2, 3, 4, 5]


In [31]:
visualize_safe_topics(ngram_models[2])

Non-outlier topics: [0, 1, 2, 3, 4, 5]


In [33]:
visualize_safe_topics(ngram_models[3])

Non-outlier topics: [0, 1, 2, 3, 4, 5]


In [25]:
base_topic_model.visualize_barchart(
    topics=base_topic_info[base_topic_info["Topic"] != -1]["Topic"].tolist(),
    top_n_topics=min(10, len(base_topic_info)),
)

In [26]:
base_topic_model.visualize_hierarchy()

In [27]:
base_topic_model.visualize_heatmap()

## Export results

In [28]:
export_outputs(
    df_with_topics=df_with_topics,
    topic_summary=scored_topic_summary_with_terms,
    ngram_terms=exact_ngram_terms,
    recommendations=recommendations,
    output_dir="./topic_outputs",
)

Saved outputs to: C:\univ\Data mining\Project\notebooks\topic_outputs


## How to adapt this notebook to any similar dataset

1. Change `data_path`.
2. Change `text_col`.
3. If your hashtag column has another name, change `hashtags_col`.
4. If the dataset has different metrics, the recommendation logic automatically looks for available numeric columns such as views, likes, comments, engagement, rates, shares, saves, impressions, and reach.
5. Use `topic_ngram_labels` to name each topic with exact 1-, 2-, and 3-gram evidence.
6. Use `recommendations` as the decision table. It is intentionally dynamic rather than hard-coded to Vanilla.

# Production Business Insight Layer (BERTopic)

This section extends the base notebook into a production-style business insight workflow with:
- resilient text/KPI detection,
- BERTopic fit/load,
- business labeling,
- KPI benchmarking,
- insight cards,
- visualization exports,
- OpenAI-assisted business explanations,
- new-post inference.


In [ ]:
# Optional install cell (run only if needed)
# %pip install -U openai bertopic sentence-transformers pandas numpy matplotlib
# %pip install -U plotly kaleido


In [ ]:
import os
import re
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

TOPIC_MODEL_DIR = "saved_bertopic_business_model"
TOPIC_LABELS_PATH = "topic_business_labels.csv"
TOPIC_KPI_PATH = "topic_kpi_benchmarks.csv"
INSIGHTS_OUTPUT_PATH = "topic_business_insights.csv"
TOP2_INSIGHTS_PATH = "top2_topic_recommendations.csv"

TEXT_CANDIDATES = ["caption_text", "caption", "text", "post_text"]


def load_env_file(env_path: str = ".env"):
    p = Path(env_path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = v


def detect_text_column(df: pd.DataFrame):
    for col in TEXT_CANDIDATES:
        if col in df.columns:
            return col
    fallback = [c for c in df.columns if "text" in c.lower() or "caption" in c.lower()]
    return fallback[0] if fallback else None


def safe_numeric(df: pd.DataFrame, col: str):
    if col not in df.columns:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[col], errors="coerce")


def get_available_kpi_columns(df: pd.DataFrame):
    candidates = [
        "engagement_rate",
        "view_rate",
        "comments_count",
        "likes_count",
        "views_count",
        "hashtags_count",
        "discount_percent",
        "post_type",
        "language",
        "CTA_present",
        "promo_post",
    ]
    return [c for c in candidates if c in df.columns]


def clean_captions(series: pd.Series):
    s = series.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s[s.str.len() >= 3]
    return s


def get_openai_client():
    load_env_file(".env")
    load_env_file("../.env")
    api_key = os.getenv("OPENAI_API_KEY", "").strip()
    if not api_key:
        print("WARNING: OPENAI_API_KEY is not set. OpenAI-based labeling/insights will use rule-based fallback.")
        return None
    if OpenAI is None:
        print("WARNING: openai SDK is not available. Install `openai` to enable LLM labeling/insights.")
        return None
    try:
        return OpenAI()
    except Exception as exc:
        print(f"WARNING: Could not initialize OpenAI client: {exc}")
        return None


def call_openai_json(client, system_prompt: str, user_prompt: str):
    if client is None:
        return None
    for model_name in ["gpt-5-mini", "gpt-4.1-mini", "gpt-4o-mini"]:
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
                response_format={"type": "json_object"},
            )
            return json.loads(resp.choices[0].message.content)
        except Exception:
            continue
    return None


def load_business_df_from_context_or_disk():
    if "df" in globals() and isinstance(globals()["df"], pd.DataFrame) and not globals()["df"].empty:
        return globals()["df"].copy()
    candidate_paths = [
        "../data/vanilla_processed.json",
        "dataset_with_topics.csv",
        "../data/business_posts.csv",
        "../data/social_media_posts.csv",
        "../data/processed_posts.csv",
    ]
    for pp in candidate_paths:
        path = Path(pp)
        if not path.exists():
            continue
        try:
            tmp = pd.read_json(path) if path.suffix.lower() == ".json" else pd.read_csv(path)
            if not tmp.empty:
                print(f"Loaded dataset from: {path}")
                return tmp
        except Exception:
            continue
    raise FileNotFoundError("No dataset found. Ensure `df` exists in memory or place a dataset in known paths.")


def count_non_outlier_topics(model: BERTopic):
    info = model.get_topic_info()
    if info.empty or "Topic" not in info.columns:
        return 0
    return int((info["Topic"] != -1).sum())


def get_topic_target_range(n_docs: int):
    n_docs = max(int(n_docs), 1)
    min_topics = max(2, min(8, int(np.sqrt(n_docs) // 2)))
    max_topics = max(min_topics + 2, min(35, int(np.sqrt(n_docs) * 2.2)))
    return min_topics, max_topics


def get_min_topic_size_candidates(n_docs: int):
    n_docs = max(int(n_docs), 1)
    candidates = [
        max(2, int(n_docs * 0.14)),
        max(2, int(n_docs * 0.10)),
        max(2, int(n_docs * 0.07)),
        max(2, int(n_docs * 0.05)),
        max(2, int(n_docs * 0.03)),
    ]
    ordered = []
    for c in candidates:
        if c not in ordered:
            ordered.append(c)
    return ordered


def fit_topic_model_adaptive(texts, embedding_model, verbose=True):
    n_docs = len(texts)
    min_topics, max_topics = get_topic_target_range(n_docs)
    target_mid = (min_topics + max_topics) / 2.0
    candidates = get_min_topic_size_candidates(n_docs)

    print(f"Adaptive target topics range: [{min_topics}, {max_topics}] for {n_docs} posts")
    print(f"Trying min_topic_size candidates: {candidates}")

    fitted = []
    for mts in candidates:
        model = BERTopic(
            embedding_model=embedding_model,
            calculate_probabilities=True,
            verbose=verbose,
            min_topic_size=mts,
        )
        _ = model.fit_transform(texts)
        k = count_non_outlier_topics(model)
        fitted.append((mts, k, model))
        print(f"Candidate min_topic_size={mts} -> {k} non-outlier topics")
        if min_topics <= k <= max_topics:
            print("Selected candidate within target range.")
            return model

    best = min(fitted, key=lambda x: abs(x[1] - target_mid))
    best_mts, best_k, best_model = best
    print(f"No exact-range candidate; selected closest: min_topic_size={best_mts}, topics={best_k}")

    if best_k > max_topics:
        print(f"Too many topics ({best_k}). Reducing to {max_topics}.")
        try:
            best_model.reduce_topics(texts, nr_topics=max_topics)
            print(f"Reduced topics -> {count_non_outlier_topics(best_model)}")
        except Exception as exc:
            print(f"Topic reduction failed: {exc}")

    return best_model



## SECTION 1 — Fit BERTopic on business corpus


In [ ]:
business_df = load_business_df_from_context_or_disk()
text_col = detect_text_column(business_df)
if text_col is None:
    raise ValueError(f"No text column found. Tried: {TEXT_CANDIDATES}")

print(f"Detected text column: {text_col}")
print(f"Available KPI columns: {get_available_kpi_columns(business_df)}")

cleaned_series = clean_captions(business_df[text_col])
business_df = business_df.loc[cleaned_series.index].copy()
business_df["_clean_text"] = cleaned_series.values
texts = business_df["_clean_text"].tolist()

if len(texts) < 10:
    print("WARNING: Very small corpus (<10 docs). Topic quality may be weak.")

embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(embedding_model_name)

model_dir = Path(TOPIC_MODEL_DIR)
min_topics_target, max_topics_target = get_topic_target_range(len(texts))

if model_dir.exists():
    try:
        topic_model = BERTopic.load(str(model_dir))
        loaded_k = count_non_outlier_topics(topic_model)
        print(f"Loaded BERTopic model from {TOPIC_MODEL_DIR} with {loaded_k} non-outlier topics")
        if loaded_k < min_topics_target or loaded_k > max_topics_target:
            print("Loaded model outside desired topic range; refitting adaptively...")
            topic_model = fit_topic_model_adaptive(texts, embedding_model, verbose=True)
            topic_model.save(
                str(model_dir),
                serialization="safetensors",
                save_ctfidf=True,
                save_embedding_model=embedding_model_name,
            )
            print(f"Saved adapted BERTopic model to {TOPIC_MODEL_DIR}")
    except Exception as exc:
        print(f"Model load failed ({exc}), refitting adaptively...")
        topic_model = fit_topic_model_adaptive(texts, embedding_model, verbose=True)
        topic_model.save(
            str(model_dir),
            serialization="safetensors",
            save_ctfidf=True,
            save_embedding_model=embedding_model_name,
        )
        print(f"Saved BERTopic model to {TOPIC_MODEL_DIR}")
else:
    topic_model = fit_topic_model_adaptive(texts, embedding_model, verbose=True)
    topic_model.save(
        str(model_dir),
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=embedding_model_name,
    )
    print(f"Saved BERTopic model to {TOPIC_MODEL_DIR}")

topic_info = topic_model.get_topic_info()
print(topic_info.head(20))
topic_info.to_csv("topic_info_raw.csv", index=False)
print("Saved raw topic info -> topic_info_raw.csv")
print(f"Final non-outlier topic count: {count_non_outlier_topics(topic_model)}")



## SECTION 2 — Convert raw topics into business labels


In [ ]:
def build_topic_summary(topic_model: BERTopic, topic_info: pd.DataFrame, top_n_words: int = 10, rep_docs_n: int = 3):
    records = []
    for _, row in topic_info.iterrows():
        topic_id = int(row["Topic"])
        if topic_id == -1:
            continue
        topic_terms = topic_model.get_topic(topic_id) or []
        words = [w for w, _ in topic_terms[:top_n_words]]
        rep_docs = topic_model.get_representative_docs(topic_id) or []
        rep_docs = [str(x).strip() for x in rep_docs[:rep_docs_n] if str(x).strip()]
        records.append(
            {
                "topic_id": topic_id,
                "topic_name_raw": str(row.get("Name", "")),
                "top_words": ", ".join(words),
                "representative_docs_sample": " || ".join(rep_docs),
                "count": int(row.get("Count", 0)),
            }
        )
    return pd.DataFrame(records)


def generate_business_label_with_openai(client, topic_row: pd.Series):
    system_prompt = (
        "You are a senior marketing strategist. Convert raw BERTopic words into a business-friendly content theme label. "
        "Return JSON only with keys: business_label, short_description, content_strategy_meaning."
    )
    user_prompt = f"""
Topic ID: {topic_row['topic_id']}
Top words: {topic_row['top_words']}
Representative docs: {topic_row['representative_docs_sample']}
Post count: {topic_row['count']}

Rules:
- Use concise marketing language.
- Avoid keyword-list labels.
- Label should be 2-5 words and dashboard-friendly.
- Consider Arabic and English context.
"""
    out = call_openai_json(client, system_prompt, user_prompt)
    if not out:
        return None
    return {
        "business_label": str(out.get("business_label", "")).strip(),
        "short_description": str(out.get("short_description", "")).strip(),
        "content_strategy_meaning": str(out.get("content_strategy_meaning", "")).strip(),
    }


def fallback_business_label(topic_row: pd.Series):
    words = [w.strip() for w in str(topic_row.get("top_words", "")).split(",") if w.strip()]
    label = f"Theme Topic {topic_row['topic_id']}" if not words else " ".join(w.title() for w in words[:2])
    return {
        "business_label": label[:60],
        "short_description": "Keyword-driven content theme detected from BERTopic terms.",
        "content_strategy_meaning": "Test this theme with clear visuals and CTA; benchmark against top-performing themes.",
    }


def deduplicate_labels(labels_df: pd.DataFrame):
    seen, deduped = {}, []
    for _, r in labels_df.iterrows():
        base = str(r["business_label"]).strip() or f"Theme Topic {int(r['topic_id'])}"
        count = seen.get(base.lower(), 0)
        seen[base.lower()] = count + 1
        deduped.append(base if count == 0 else f"{base} (Topic {int(r['topic_id'])})")
    labels_df = labels_df.copy()
    labels_df["business_label"] = deduped
    return labels_df


def load_or_create_topic_labels(topic_model, topic_info, labels_path=TOPIC_LABELS_PATH):
    summary_df = build_topic_summary(topic_model, topic_info)
    client = get_openai_client()
    existing = pd.read_csv(labels_path) if Path(labels_path).exists() else None
    if existing is not None:
        print(f"Loaded existing labels file: {labels_path}")

    records = []
    for _, row in summary_df.iterrows():
        topic_id = int(row["topic_id"])
        if existing is not None:
            existing_row = existing.loc[existing["topic_id"] == topic_id]
            if not existing_row.empty:
                ex = existing_row.iloc[0]
                if str(ex.get("business_label", "")).strip():
                    records.append(
                        {
                            **row.to_dict(),
                            "business_label": ex.get("business_label", ""),
                            "short_description": ex.get("short_description", ""),
                            "content_strategy_meaning": ex.get("content_strategy_meaning", ""),
                        }
                    )
                    continue

        generated = generate_business_label_with_openai(client, row) or fallback_business_label(row)
        records.append({**row.to_dict(), **generated})

    labels_df = pd.DataFrame(records)
    outlier_count = int(topic_info.loc[topic_info["Topic"] == -1, "Count"].iloc[0]) if (topic_info["Topic"] == -1).any() else 0
    outlier_row = pd.DataFrame(
        [
            {
                "topic_id": -1,
                "topic_name_raw": "-1",
                "top_words": "",
                "representative_docs_sample": "",
                "count": outlier_count,
                "business_label": "Outlier / Mixed Content",
                "short_description": "Mixed or noisy posts that do not map cleanly to a single strategy theme.",
                "content_strategy_meaning": "Review manually; avoid using as a standalone strategic content pillar.",
            }
        ]
    )

    labels_df = pd.concat([labels_df, outlier_row], ignore_index=True)
    labels_df = deduplicate_labels(labels_df)
    labels_df.to_csv(labels_path, index=False)
    print(f"Saved business topic labels -> {labels_path}")
    return labels_df


topic_labels_df = load_or_create_topic_labels(topic_model, topic_info, TOPIC_LABELS_PATH)
display(topic_labels_df.head(20))


## SECTION 3 — Assign every post to a topic


In [ ]:
topics, probs = topic_model.transform(texts)

business_df = business_df.copy()
business_df["topic_id"] = topics

if probs is None:
    business_df["topic_probability"] = np.nan
else:
    topic_probability = []
    for i, t in enumerate(topics):
        try:
            p = float(probs[i][t]) if t >= 0 else float(np.max(probs[i]))
        except Exception:
            p = np.nan
        topic_probability.append(p)
    business_df["topic_probability"] = topic_probability

label_map = topic_labels_df.set_index("topic_id")["business_label"].to_dict()
desc_map = topic_labels_df.set_index("topic_id")["short_description"].to_dict()

business_df["business_label"] = business_df["topic_id"].map(label_map).fillna("Unknown Theme")
business_df["topic_description"] = business_df["topic_id"].map(desc_map).fillna("")

business_df.to_csv("posts_with_business_topics.csv", index=False)
print("Saved enriched posts -> posts_with_business_topics.csv")

print("Topic ID counts:")
print(business_df["topic_id"].value_counts(dropna=False).head(20))
print("\nBusiness label counts:")
print(business_df["business_label"].value_counts(dropna=False).head(20))

sample_col = text_col
sample_posts = (
    business_df[["business_label", sample_col]]
    .dropna()
    .groupby("business_label", as_index=False)
    .head(2)
    .rename(columns={sample_col: "sample_post"})
)
print("\nSample posts per business label:")
display(sample_posts.head(40))


## SECTION 4 — Aggregate KPIs per topic


In [ ]:
def aggregate_topic_kpis(df: pd.DataFrame, text_col: str):
    work = df.copy()
    numeric_cols = [
        "engagement_rate",
        "view_rate",
        "comments_count",
        "likes_count",
        "views_count",
        "hashtags_count",
        "discount_percent",
    ]
    for c in numeric_cols:
        if c in work.columns:
            work[c] = safe_numeric(work, c)

    rows = []
    for label, grp in work.groupby("business_label", dropna=False):
        row = {
            "business_label": label,
            "post_count": int(len(grp)),
            "dominant_topic_id": int(grp["topic_id"].mode(dropna=True).iloc[0]) if grp["topic_id"].notna().any() else np.nan,
            "topic_id_list": ", ".join(sorted({str(int(x)) for x in grp["topic_id"].dropna().tolist()})) if grp["topic_id"].notna().any() else "",
            "representative_posts": " || ".join(grp[text_col].astype(str).head(2).tolist()),
        }

        if "engagement_rate" in grp:
            row["avg_engagement_rate"] = grp["engagement_rate"].mean()
            row["median_engagement_rate"] = grp["engagement_rate"].median()
        if "view_rate" in grp:
            row["avg_view_rate"] = grp["view_rate"].mean()
            row["median_view_rate"] = grp["view_rate"].median()
        if "comments_count" in grp:
            row["avg_comments_count"] = grp["comments_count"].mean()
        if "likes_count" in grp:
            row["avg_likes_count"] = grp["likes_count"].mean()
        if "views_count" in grp:
            row["avg_views_count"] = grp["views_count"].mean()
        if "hashtags_count" in grp:
            row["avg_hashtags_count"] = grp["hashtags_count"].mean()
        if "discount_percent" in grp:
            row["avg_discount_percent"] = grp["discount_percent"].mean()

        if "post_type" in grp.columns:
            row["percentage_reels"] = (grp["post_type"].astype(str).str.lower() == "reel").mean() * 100
        if "promo_post" in grp.columns:
            promo = grp["promo_post"].astype(str).str.lower().isin(["1", "true", "yes", "y"])
            row["percentage_promo_posts"] = promo.mean() * 100
        if "CTA_present" in grp.columns:
            cta = grp["CTA_present"].astype(str).str.lower().isin(["1", "true", "yes", "y"])
            row["percentage_CTA_posts"] = cta.mean() * 100

        rows.append(row)

    kpi_df = pd.DataFrame(rows)
    sort_priority = ["avg_engagement_rate", "avg_view_rate", "avg_views_count", "post_count"]
    sort_col = next((c for c in sort_priority if c in kpi_df.columns), "post_count")
    return kpi_df.sort_values(sort_col, ascending=False).reset_index(drop=True), sort_col


def aggregate_topic_kpis_by_topic(df: pd.DataFrame):
    work = df.copy()
    for c in ["engagement_rate", "view_rate", "views_count"]:
        if c in work.columns:
            work[c] = safe_numeric(work, c)

    rows = []
    for topic_id, grp in work.groupby("topic_id", dropna=False):
        if pd.isna(topic_id):
            continue
        row = {
            "topic_id": int(topic_id),
            "business_label": grp["business_label"].mode().iloc[0] if "business_label" in grp.columns and not grp["business_label"].isna().all() else "Unknown Theme",
            "post_count": int(len(grp)),
        }
        if "engagement_rate" in grp:
            row["avg_engagement_rate"] = grp["engagement_rate"].mean()
        if "view_rate" in grp:
            row["avg_view_rate"] = grp["view_rate"].mean()
        if "views_count" in grp:
            row["avg_views_count"] = grp["views_count"].mean()
        rows.append(row)

    topic_df = pd.DataFrame(rows)
    if topic_df.empty:
        return topic_df

    sort_col = "avg_engagement_rate" if "avg_engagement_rate" in topic_df.columns else "post_count"
    topic_df = topic_df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    return topic_df


topic_kpi_df, ranking_metric = aggregate_topic_kpis(business_df, text_col)
topic_kpi_by_topic_df = aggregate_topic_kpis_by_topic(business_df)
topic_kpi_df.to_csv(TOPIC_KPI_PATH, index=False)

print(f"Sorted by: {ranking_metric}")
print(f"Saved KPI benchmarks -> {TOPIC_KPI_PATH}")
display(topic_kpi_df)
print("Topic-level KPI preview:")
display(topic_kpi_by_topic_df.head(20))



## SECTION 5 — Generate business insight cards


In [ ]:
def generate_rule_based_insights(kpi_df: pd.DataFrame):
    insights = []
    if kpi_df.empty:
        return pd.DataFrame(
            columns=["insight_title", "finding", "evidence", "recommendation", "priority", "metric_used"]
        )

    metric = next(
        (c for c in ["avg_engagement_rate", "avg_view_rate", "avg_views_count", "post_count"] if c in kpi_df.columns),
        "post_count",
    )
    top = kpi_df.sort_values(metric, ascending=False).iloc[0]
    bottom = kpi_df.sort_values(metric, ascending=True).iloc[0]

    insights.append(
        {
            "insight_title": "Top Performing Theme",
            "finding": f"{top['business_label']} is the highest performing theme by {metric}.",
            "evidence": f"{metric}={top[metric]:.4f}" if pd.notna(top[metric]) else f"{metric} unavailable",
            "recommendation": "Scale this theme with consistent formats and stronger CTA testing.",
            "priority": "High",
            "metric_used": metric,
        }
    )

    if len(kpi_df) > 1:
        insights.append(
            {
                "insight_title": "Underperforming Theme",
                "finding": f"{bottom['business_label']} is the lowest performing theme by {metric}.",
                "evidence": f"{metric}={bottom[metric]:.4f}" if pd.notna(bottom[metric]) else f"{metric} unavailable",
                "recommendation": "Reposition messaging, hook, and creative format before scaling budget.",
                "priority": "High",
                "metric_used": metric,
            }
        )

    if len(kpi_df) > 1 and pd.notna(top.get(metric)) and pd.notna(bottom.get(metric)) and float(bottom[metric]) != 0:
        ratio = float(top[metric]) / float(bottom[metric])
        insights.append(
            {
                "insight_title": "Performance Gap",
                "finding": f"Top theme outperforms lowest theme by {ratio:.2f}x on {metric}.",
                "evidence": f"{top['business_label']} vs {bottom['business_label']}",
                "recommendation": "Shift content mix toward top themes while redesigning weak themes.",
                "priority": "Medium",
                "metric_used": metric,
            }
        )

    if "percentage_CTA_posts" in kpi_df.columns and "avg_comments_count" in kpi_df.columns:
        cta_top = kpi_df.sort_values("percentage_CTA_posts", ascending=False).iloc[0]
        insights.append(
            {
                "insight_title": "CTA Pattern",
                "finding": f"{cta_top['business_label']} has the highest CTA share.",
                "evidence": f"CTA%={cta_top['percentage_CTA_posts']:.1f}, Avg comments={cta_top.get('avg_comments_count', np.nan):.2f}",
                "recommendation": "Balance direct CTA posts with community-oriented captions to improve discussion depth.",
                "priority": "Medium",
                "metric_used": "percentage_CTA_posts",
            }
        )

    return pd.DataFrame(insights)


def select_top2_topics(topic_kpi_by_topic_df: pd.DataFrame):
    if topic_kpi_by_topic_df.empty:
        return topic_kpi_by_topic_df

    df = topic_kpi_by_topic_df.copy()
    use_view_col = "avg_view_rate" if "avg_view_rate" in df.columns else ("avg_views_count" if "avg_views_count" in df.columns else None)

    if "avg_engagement_rate" in df.columns:
        eng_rank = df["avg_engagement_rate"].rank(pct=True, method="average")
    else:
        eng_rank = pd.Series([0.0] * len(df), index=df.index)

    if use_view_col is not None:
        view_rank = df[use_view_col].rank(pct=True, method="average")
    else:
        view_rank = pd.Series([0.0] * len(df), index=df.index)

    df["combined_score"] = (eng_rank + view_rank) / 2.0
    return df.sort_values(["combined_score", "post_count"], ascending=[False, False]).head(2).reset_index(drop=True)


def generate_top2_recommendations_with_openai(top2_df: pd.DataFrame, topic_labels_df: pd.DataFrame, client=None):
    if top2_df.empty:
        return pd.DataFrame(columns=[
            "topic_id", "business_label", "top_words", "avg_engagement_rate", "avg_view_rate", "avg_views_count", "combined_score", "recommendation"
        ])

    word_map = topic_labels_df.set_index("topic_id")["top_words"].to_dict() if "top_words" in topic_labels_df.columns else {}
    out_rows = []

    for _, r in top2_df.iterrows():
        topic_id = int(r["topic_id"])
        top_words = str(word_map.get(topic_id, ""))
        payload = {
            "topic_id": topic_id,
            "business_label": str(r.get("business_label", "")),
            "top_10_words": top_words,
            "avg_engagement_rate": None if pd.isna(r.get("avg_engagement_rate", np.nan)) else float(r.get("avg_engagement_rate")),
            "avg_view_rate": None if pd.isna(r.get("avg_view_rate", np.nan)) else float(r.get("avg_view_rate")),
            "avg_views_count": None if pd.isna(r.get("avg_views_count", np.nan)) else float(r.get("avg_views_count")),
            "combined_score": None if pd.isna(r.get("combined_score", np.nan)) else float(r.get("combined_score")),
        }

        recommendation_text = ""
        if client is not None:
            result = call_openai_json(
                client,
                "You are a social media strategy expert. Return JSON with one key: recommendation. Keep it concise and actionable.",
                "Create a recommendation for this top-performing topic based on its words and KPIs: " + json.dumps(payload, ensure_ascii=False),
            )
            if result and str(result.get("recommendation", "")).strip():
                recommendation_text = str(result.get("recommendation")).strip()

        if not recommendation_text:
            recommendation_text = (
                "High-performing topic. Reuse this theme with similar wording, maintain visual consistency, "
                "and test CTA variants to compound engagement and view performance."
            )

        out_rows.append({
            "topic_id": topic_id,
            "business_label": str(r.get("business_label", "")),
            "top_words": top_words,
            "avg_engagement_rate": r.get("avg_engagement_rate", np.nan),
            "avg_view_rate": r.get("avg_view_rate", np.nan),
            "avg_views_count": r.get("avg_views_count", np.nan),
            "combined_score": r.get("combined_score", np.nan),
            "recommendation": recommendation_text,
        })

    return pd.DataFrame(out_rows)


def rewrite_insights_with_openai(insights_df: pd.DataFrame, client=None):
    if insights_df.empty or client is None:
        return insights_df
    out = call_openai_json(
        client,
        "You are a marketing analytics assistant. Rewrite each insight into concise dashboard card copy. Return JSON with key cards.",
        json.dumps({"cards": insights_df.to_dict(orient="records")}, ensure_ascii=False),
    )
    if not out or "cards" not in out:
        return insights_df
    try:
        rewritten = pd.DataFrame(out["cards"])
        needed = ["insight_title", "finding", "evidence", "recommendation", "priority", "metric_used"]
        for c in needed:
            if c not in rewritten.columns:
                rewritten[c] = insights_df[c] if c in insights_df.columns else ""
        return rewritten[needed]
    except Exception:
        return insights_df


openai_client = get_openai_client()
insights_df = generate_rule_based_insights(topic_kpi_df)

top2_topics_df = select_top2_topics(topic_kpi_by_topic_df)
top2_reco_df = generate_top2_recommendations_with_openai(top2_topics_df, topic_labels_df, openai_client)
top2_reco_df.to_csv(TOP2_INSIGHTS_PATH, index=False)
print(f"Saved top-2 topic recommendations -> {TOP2_INSIGHTS_PATH}")
display(top2_reco_df)

if not top2_reco_df.empty:
    extra_cards = []
    for _, rr in top2_reco_df.iterrows():
        topic_metric_text = ""
        if pd.notna(rr.get("avg_engagement_rate", np.nan)):
            topic_metric_text += f"avg_engagement_rate={rr['avg_engagement_rate']:.4f}; "
        if pd.notna(rr.get("avg_view_rate", np.nan)):
            topic_metric_text += f"avg_view_rate={rr['avg_view_rate']:.4f}; "
        elif pd.notna(rr.get("avg_views_count", np.nan)):
            topic_metric_text += f"avg_views_count={rr['avg_views_count']:.2f}; "

        extra_cards.append({
            "insight_title": f"Top Topic Recommendation: {rr['business_label']}",
            "finding": f"Topic {int(rr['topic_id'])} is in the top 2 by combined engagement/view score.",
            "evidence": topic_metric_text.strip(),
            "recommendation": rr["recommendation"],
            "priority": "High",
            "metric_used": "combined_engagement_view_score",
        })

    insights_df = pd.concat([insights_df, pd.DataFrame(extra_cards)], ignore_index=True)

insights_df = rewrite_insights_with_openai(insights_df, openai_client)
insights_df.to_csv(INSIGHTS_OUTPUT_PATH, index=False)

print(f"Saved insights -> {INSIGHTS_OUTPUT_PATH}")
for _, row in insights_df.iterrows():
    print("\n---")
    print(f"### {row['insight_title']}")
    print(f"- Finding: {row['finding']}")
    print(f"- Evidence: {row['evidence']}")
    print(f"- Recommendation: {row['recommendation']}")
    print(f"- Priority: {row['priority']}")
    print(f"- Metric: {row['metric_used']}")



## SECTION 6 — Visualizations


In [ ]:
def save_horizontal_bar(df: pd.DataFrame, value_col: str, out_path: str, title: str, label_col: str = "business_label"):
    if value_col not in df.columns or df.empty:
        print(f"Skipped {out_path}: missing {value_col}")
        return
    plot_df = df[[label_col, value_col]].dropna().sort_values(value_col, ascending=False)
    if plot_df.empty:
        print(f"Skipped {out_path}: no data")
        return
    fig_h = max(4, 0.45 * len(plot_df))
    fig, ax = plt.subplots(figsize=(10, fig_h))
    labels_wrapped = [textwrap.fill(str(x), width=35) for x in plot_df[label_col]]
    ax.barh(labels_wrapped[::-1], plot_df[value_col].values[::-1])
    ax.set_title(title)
    ax.set_xlabel(value_col)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")


def save_topic_visualizations(topic_model, kpi_df: pd.DataFrame):
    try:
        fig = topic_model.visualize_topics()
        fig.write_html("01_intertopic_distance_map.html")
        print("Saved 01_intertopic_distance_map.html")
        try:
            fig.write_image("01_intertopic_distance_map.png")
            print("Saved 01_intertopic_distance_map.png")
        except Exception as exc:
            print(f"Could not save intertopic PNG (install kaleido if needed): {exc}")
    except Exception as exc:
        print(f"Could not generate intertopic map: {exc}")

    save_horizontal_bar(kpi_df, "avg_engagement_rate", "02_avg_engagement_by_topic.png", "Average Engagement Rate by Business Topic")
    if "avg_view_rate" in kpi_df.columns:
        save_horizontal_bar(kpi_df, "avg_view_rate", "03_avg_views_by_topic.png", "Average View Rate by Business Topic")
    elif "avg_views_count" in kpi_df.columns:
        save_horizontal_bar(kpi_df, "avg_views_count", "03_avg_views_by_topic.png", "Average Views by Business Topic")
    else:
        print("Skipped 03_avg_views_by_topic.png: missing view metrics")
    save_horizontal_bar(kpi_df, "post_count", "04_post_count_by_topic.png", "Post Count by Business Topic")
    save_horizontal_bar(kpi_df, "avg_comments_count", "05_avg_comments_by_topic.png", "Average Comments by Business Topic")


save_topic_visualizations(topic_model, topic_kpi_df)


## SECTION 7 — OpenAI visual interpretation


In [ ]:
def generate_business_explanation(topic_labels_df, topic_kpi_df, insights_df, client=None):
    chart_paths = [
        "01_intertopic_distance_map.html",
        "01_intertopic_distance_map.png",
        "02_avg_engagement_by_topic.png",
        "03_avg_views_by_topic.png",
        "04_post_count_by_topic.png",
        "05_avg_comments_by_topic.png",
    ]
    existing_charts = [p for p in chart_paths if Path(p).exists()]
    payload = {
        "topic_labels": topic_labels_df.head(50).to_dict(orient="records"),
        "kpi_benchmarks": topic_kpi_df.head(50).to_dict(orient="records"),
        "insights": insights_df.to_dict(orient="records"),
        "chart_files_saved": existing_charts,
        "note": "Local images are saved on disk. If image encoding is not configured, use these tables and metrics.",
    }

    if client is None:
        md = """
# Topic Cluster Business Explanation

OpenAI client not available, so this explanation is rule-based.

- Topic clusters represent recurring content themes in your captions.
- Higher KPI themes are stronger strategic pillars to scale.
- Lower KPI themes should be redesigned (hook, offer framing, visuals, CTA).
- Intertopic distance map interpretation:
  - Close clusters: semantically similar themes.
  - Far clusters: distinct themes.
  - Large bubbles: higher post volume.
  - `Outlier / Mixed Content`: not a clean strategic segment.
""".strip()
        Path("topic_cluster_business_explanation.md").write_text(md, encoding="utf-8")
        return md

    out = None
    for model_name in ["gpt-5-mini", "gpt-4.1-mini", "gpt-4o-mini"]:
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a senior marketing analyst. Create concise markdown explanation for stakeholders.",
                    },
                    {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
                ],
                temperature=0.2,
            )
            out = resp.choices[0].message.content
            break
        except Exception:
            continue

    if not out:
        out = """
# Topic Cluster Business Explanation

OpenAI explanation generation failed, so this is rule-based fallback.

- Use KPI leaders as core content pillars.
- Improve weak themes before scaling.
- Intertopic map: close=similar, far=distinct, large bubbles=more posts, outlier topic is mixed content.
""".strip()

    Path("topic_cluster_business_explanation.md").write_text(out, encoding="utf-8")
    return out


business_explanation_md = generate_business_explanation(topic_labels_df, topic_kpi_df, insights_df, openai_client)
print("Saved -> topic_cluster_business_explanation.md")
print("\nPreview:\n")
print(business_explanation_md[:3000])



## SECTION 8 — Production-style inference function


In [ ]:
def assign_topic_to_new_posts(new_posts, topic_model, label_map, kpi_benchmarks):
    if not isinstance(new_posts, (list, tuple, pd.Series)):
        raise ValueError("new_posts must be a list-like object of captions")

    clean_posts = pd.Series(new_posts).fillna("").astype(str).str.strip()
    clean_posts = clean_posts[clean_posts.str.len() > 0]
    if clean_posts.empty:
        return pd.DataFrame(
            columns=[
                "post_text",
                "topic_id",
                "topic_probability",
                "business_label",
                "benchmark_engagement_rate",
                "benchmark_view_rate",
                "recommendation",
            ]
        )

    topics, probs = topic_model.transform(clean_posts.tolist())
    pred = pd.DataFrame({"post_text": clean_posts.tolist(), "topic_id": topics})

    if probs is not None:
        conf = []
        for i, t in enumerate(topics):
            try:
                conf.append(float(probs[i][t]) if t >= 0 else float(np.max(probs[i])))
            except Exception:
                conf.append(np.nan)
        pred["topic_probability"] = conf
    else:
        pred["topic_probability"] = np.nan

    pred["business_label"] = pred["topic_id"].map(label_map).fillna("Unknown Theme")

    bench = kpi_benchmarks.copy()
    keep_cols = ["business_label"]
    if "avg_engagement_rate" in bench.columns:
        keep_cols.append("avg_engagement_rate")
    if "avg_view_rate" in bench.columns:
        keep_cols.append("avg_view_rate")
    bench = bench[keep_cols].drop_duplicates(subset=["business_label"])

    pred = pred.merge(bench, on="business_label", how="left").rename(
        columns={
            "avg_engagement_rate": "benchmark_engagement_rate",
            "avg_view_rate": "benchmark_view_rate",
        }
    )

    overall_eng = (
        pd.to_numeric(kpi_benchmarks["avg_engagement_rate"], errors="coerce").mean()
        if "avg_engagement_rate" in kpi_benchmarks.columns
        else np.nan
    )
    recs = []
    for _, r in pred.iterrows():
        eng = r.get("benchmark_engagement_rate", np.nan)
        if pd.notna(eng) and pd.notna(overall_eng):
            if eng >= overall_eng:
                recs.append("This topic performs above average. Use similar branding and add a clear CTA.")
            else:
                recs.append("This topic is below average. Test stronger hooks, tighter visuals, and revised CTA.")
        else:
            recs.append("No benchmark available. Publish as experiment and monitor early KPI response.")

    pred["recommendation"] = recs
    return pred


# Example usage:
# sample_new_posts = ["Try our new vanilla iced coffee", "رمضان كريم مع عروض اليوم"]
# infer_df = assign_topic_to_new_posts(sample_new_posts, topic_model, label_map, topic_kpi_df)
# display(infer_df)


## SECTION 9 — Final notebook summary


In [ ]:
num_posts = len(business_df)
num_topics = int((topic_info["Topic"] != -1).sum()) if "Topic" in topic_info.columns else int(topic_info.shape[0])
num_labels = topic_labels_df["business_label"].nunique() if "business_label" in topic_labels_df.columns else 0

best_topic = "N/A"
worst_topic = "N/A"
if not topic_kpi_df.empty:
    metric_candidates = [c for c in ["avg_engagement_rate", "avg_view_rate", "avg_views_count", "post_count"] if c in topic_kpi_df.columns]
    metric_use = metric_candidates[0] if metric_candidates else "post_count"
    best_topic = str(topic_kpi_df.sort_values(metric_use, ascending=False).iloc[0]["business_label"])
    worst_topic = str(topic_kpi_df.sort_values(metric_use, ascending=True).iloc[0]["business_label"])
else:
    metric_use = "N/A"

print("=" * 80)
print("BUSINESS TOPIC DASHBOARD SUMMARY")
print("=" * 80)
print(f"Number of posts analyzed: {num_posts}")
print(f"Number of discovered topics (excluding -1): {num_topics}")
print(f"Number of business labels: {num_labels}")
print(f"Best-performing topic: {best_topic}")
print(f"Worst-performing topic: {worst_topic}")
print(f"Ranking metric used: {metric_use}")

print("\nTop 5 recommendations:")
if not insights_df.empty:
    for i, rec in enumerate(insights_df["recommendation"].head(5).tolist(), 1):
        print(f"{i}. {rec}")
else:
    print("1. No recommendations generated.")

print("\nSaved outputs:")
for pp in [
    TOPIC_MODEL_DIR,
    TOPIC_LABELS_PATH,
    TOPIC_KPI_PATH,
    "posts_with_business_topics.csv",
    INSIGHTS_OUTPUT_PATH,
    TOP2_INSIGHTS_PATH,
    "01_intertopic_distance_map.html",
    "01_intertopic_distance_map.png",
    "02_avg_engagement_by_topic.png",
    "03_avg_views_by_topic.png",
    "04_post_count_by_topic.png",
    "05_avg_comments_by_topic.png",
    "topic_cluster_business_explanation.md",
]:
    print(f"- {pp}")

